# Reentrenamiento calibrado TinyML — ESP32-CAM
El dataset mezcla mediciones reales y **datos sintéticos** para cubrir zonas no observadas. Los datos sintéticos NO son mediciones experimentales y no se usan para demostrar exactitud real.


In [ ]:
import numpy as np, pandas as pd, tensorflow as tf, matplotlib.pyplot as plt, json, zipfile, os
from google.colab import files
print('TensorFlow:',tf.__version__)
uploaded = files.upload()
archivo = 'temperatura_humedad_calibracion.csv' if 'temperatura_humedad_calibracion.csv' in uploaded else None
assert archivo is not None, 'Sube temperatura_humedad_calibracion.csv'
df = pd.read_csv(archivo)
print(df.groupby(['origen','clase']).size())
print(df[['temperatura','humedad']].describe())
print('Muestras reales FRIO:',len(df[(df.origen=='REAL') & (df.clase=='FRIO')]))
print('ATENCION: NO hay muestras reales por debajo de 12 °C; es indispensable validar o capturar nuevas.')
assert (df.clase == np.where(df.temperatura<12,'FRIO',np.where(df.temperatura<21,'TEMPLADO','CALIENTE'))).all(), 'Etiquetas incompatibles con límites 12/21'


## EDA y cobertura térmica
Las clases se definen mediante los límites originales: frío <12 °C, templado entre 12 y <21 °C, caliente ≥21 °C.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(13,4))
for clase,g in df.groupby('clase'):
 axs[0].hist(g['temperatura'],bins=35,alpha=.5,label=clase)
 axs[1].scatter(g['temperatura'],g['humedad'],s=4,alpha=.15,label=clase)
axs[0].axvline(12,c='k',ls='--');axs[0].axvline(21,c='k',ls='--')
axs[0].set_title('Temperatura por clase');axs[1].set_title('Temperatura vs humedad')
for a in axs:a.legend()
plt.show()

## Modelo: temperatura + humedad, 16–8–3
Separación de pares únicos sintéticos para evaluación de ajuste. Separaremos las **mediciones reales** como una auditoría descriptiva, no como validación independiente, porque proceden de pocas sesiones.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
clases=['FRIO','TEMPLADO','CALIENTE']
# Una pareja solo se usa una vez en el entrenamiento.
unicos=df.drop_duplicates(subset=['temperatura','humedad']).copy()
train,val=train_test_split(unicos,test_size=.20,random_state=42,stratify=unicos['clase'])
xtrain=train[['temperatura','humedad']].to_numpy(dtype='float32')
xval=val[['temperatura','humedad']].to_numpy(dtype='float32')
ytrain=np.array([clases.index(x) for x in train['clase']],dtype='int32')
yval=np.array([clases.index(x) for x in val['clase']],dtype='int32')
mean=xtrain.mean(axis=0); std=xtrain.std(axis=0);std=np.maximum(std,1.e-6)
xtrain=(xtrain-mean)/std;xval=(xval-mean)/std
print('Media',mean,'Desv.',std)
tf.keras.utils.set_random_seed(2026)
modelo=tf.keras.Sequential([
 tf.keras.layers.Input(shape=(2,)),
 tf.keras.layers.Dense(16,activation='relu'),
 tf.keras.layers.Dense(8,activation='relu'),
 tf.keras.layers.Dense(3,activation='softmax')])
modelo.compile(optimizer=tf.keras.optimizers.Adam(0.003),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
hist=modelo.fit(xtrain,ytrain,validation_data=(xval,yval),epochs=100,batch_size=32,verbose=1,
 callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=14,restore_best_weights=True)])
plt.plot(hist.history['loss'],label='Entrenamiento');plt.plot(hist.history['val_loss'],label='Validación');plt.legend();plt.title('Pérdida');plt.show()
pred=np.argmax(modelo.predict(xval,verbose=0),axis=1)
print(classification_report(yval,pred,target_names=clases))
ConfusionMatrixDisplay(confusion_matrix(yval,pred),display_labels=clases).plot();plt.show()

## Pruebas en límites y auditoría de la red
No interpretar los datos sintéticos como medidas reales independientes.

In [ ]:
pruebas=np.array([[8,60],[11.8,75],[11.99,30],[12.01,65],[18.0,48],[20.8,60],[21.01,50],[27,75],[40,25]],dtype='float32')
ps=modelo.predict((pruebas-mean)/std,verbose=0)
for x,p in zip(pruebas,ps):print('T=',x[0],'H=',x[1],'pred=',clases[np.argmax(p)],'prob=',np.round(p,3))
# Examen de consistencia en amplio rango de humedad y temperaturas.
fail=[]
malla=np.array([(t,hum) for t in np.arange(5,43,0.2) for hum in [25,40,60,85]],dtype='float32')
pred_malla=np.argmax(modelo.predict((malla-mean)/std,verbose=0),axis=1)
for (t,hum),pred in zip(malla,pred_malla):
 ref=0 if t<12 else (1 if t<21 else 2)
 if int(pred)!=ref:fail.append((round(float(t),2),float(hum),clases[int(pred)],clases[ref]))
print('Discrepancias en malla sintética:',len(fail),'de',len(np.arange(5,43,0.2))*4)
print('Ejemplos:',fail[:15])

## Exportación TFLite + model.h
El archivo `model.h` incluye el modelo y los factores de normalización de este entrenamiento. El firmware debe recompilarse con este archivo.

In [ ]:
conv=tf.lite.TFLiteConverter.from_keras_model(modelo)
# Float32 para mantener compatibilidad con el firmware ArduTFLite existente.
tflite=conv.convert()
open('modelo_temperatura_calibrado.tflite','wb').write(tflite)
with open('model.h','w',encoding='utf8') as f:
 f.write('#pragma once\n#include <stdint.h>\n')
 f.write('alignas(16) const unsigned char modelo_temperatura[] = {\n')
 for i in range(0,len(tflite),12):
  f.write('  '+', '.join('0x%02x'%b for b in tflite[i:i+12])+',\n')
 f.write('};\n')
 f.write('const float media_entrada[2] = {'+','.join(format(float(v),'.9g') + ('f' if '.' in format(float(v),'.9g') or 'e' in format(float(v),'.9g') else '.0f') for v in mean)+'};\n')
 f.write('const float desv_entrada[2] = {'+','.join(format(float(v),'.9g') + ('f' if '.' in format(float(v),'.9g') or 'e' in format(float(v),'.9g') else '.0f') for v in std)+'};\n')
metrics={'origen':'REENTRENAMIENTO_CON_DATOS_SINTETICOS','real':int((df.origen=='REAL').sum()),'sinteticos':int((df.origen=='SINTETICO').sum()),'accuracy_val_mixta':float(np.mean(pred==yval)),'discrepancias_malla':len(fail),'nota':'No es validacion independiente en nuevas sesiones reales'}
open('metricas_calibracion.json','w').write(json.dumps(metrics,indent=2,ensure_ascii=False))
with zipfile.ZipFile('modelo_calibrado_exportado.zip','w') as z:
 for n in ['model.h','modelo_temperatura_calibrado.tflite','metricas_calibracion.json']: z.write(n)
print('Modelo bytes:',len(tflite));files.download('modelo_calibrado_exportado.zip')